# Probability of a Positive Day
By treating the daily returns as a normal distribution utilizing the `pc_mean` and `pc_std`, we can model the statistical probability that the stock will close 'green' on any randomly given trading day.

In [ ]:
import gspread
from gspread_dataframe import get_as_dataframe

# Authenticate with your Google account
gc = gspread.service_account(filename='/Users/philipmassey/.config/gspread/service_account.json')

import sys
import os
import pandas as pd
import numpy as np

# Add the parent directory to the path so we can import the project modules
sys.path.append(os.path.abspath('../../'))

import market_data as md
import performance as pf

dct_workbook_url = {
    'Portfolio Adjustments': 'https://docs.google.com/spreadsheets/d/1bTsH3cjQDGR-Mlnq-bypRqhGIHJApKKJgsgXWSemur4/edit#gid=0',
    'Dividends': 'https://docs.google.com/spreadsheets/d/1N1zyOStCH-gvYAgCnv6vtmsTkp6q7R8rpAnzwv9W1sI/edit?gid=0',
    'Relative Strength / Sector Leadership': 'https://docs.google.com/spreadsheets/d/16f_asTlYBbUpq9dkoV3Wqvc1wc71YVI4MbO7LUf7dpI/edit?gid=0',
    'Quality Of Trend Consistency': 'https://docs.google.com/spreadsheets/d/1DE95ydBHX1J5LrvuuHN3_wy1iCzYbtjfpJR28mcQwnk',
    'Probability Of Positive Day': 'https://docs.google.com/spreadsheets/d/1oSyGUY7aiBcjWlKYa5EvXFrmOUpWlasLmT94RSIeRjU',
    'Mean Reversion Rubber Band': 'https://docs.google.com/spreadsheets/d/188r11UEo0rrh-uZT-4mz1pnQj__rBOp1XyN5FuOzMRc',
    'Kelly Criterion Optimal Sizing': 'https://docs.google.com/spreadsheets/d/1rIeckdcAxFiqJDuEaClYL0uahnDddxDtEPAyyIZM1Cs'
}

dct_worksheet_ids = {
    'Relative Strength / Sector Leadership': {pf.calc_percent_daily:0, pf.calc_percent_weekly:1171511640, pf.calc_percent_2weekly:1906167718, pf.calc_percent_monthly:2087427134, pf.calc_percent_2monthly:396380362},
    'Quality Of Trend Consistency': {pf.calc_percent_daily: 0, pf.calc_percent_weekly: 368623089, pf.calc_percent_2weekly: 31392050, pf.calc_percent_monthly: 1948826216, pf.calc_percent_2monthly: 1625076267},
    'Probability Of Positive Day': {pf.calc_percent_daily: 0, pf.calc_percent_weekly: 911684053, pf.calc_percent_2weekly: 2001663835, pf.calc_percent_monthly: 707711513, pf.calc_percent_2monthly: 1808101499},
    'Mean Reversion Rubber Band': {pf.calc_percent_daily: 0, pf.calc_percent_weekly: 1645850023, pf.calc_percent_2weekly: 2020910653, pf.calc_percent_monthly: 550915282, pf.calc_percent_2monthly: 1119783595},
    'Kelly Criterion Optimal Sizing': {pf.calc_percent_daily: 0, pf.calc_percent_weekly: 891321177, pf.calc_percent_2weekly: 1723754803, pf.calc_percent_monthly: 1285903326, pf.calc_percent_2monthly: 1948635551}
}

def worksheet_update_with_df(workbook_name, worksheet_id, df):
    workbook_url = dct_workbook_url[workbook_name]
    workbook = gc.open_by_url(workbook_url)
    worksheet = workbook.get_worksheet_by_id(worksheet_id)
    # If you need to clear old data that might be longer than the new DataFrame,
    # use batch_clear() with a range. This clears VALUES but keeps FORMATTING.
    worksheet.batch_clear(["A1:Z500"])
    # Use set_with_dataframe to update values only.
    # It preserves existing formatting (colors, fonts, borders).
    result = md.set_with_dataframe(worksheet, df, row=1, col=1, include_index=False, include_column_header=True)
    print('\tupdated', worksheet.title, '\t',result)


ServerSelectionTimeoutError: connection closed, Timeout: 10.0s, Topology Description: <TopologyDescription id: 69bdf72319e170feadbb7728, topology_type: Single, servers: [<ServerDescription ('localhost', 27017) server_type: Unknown, rtt: None, error=AutoReconnect('connection closed')>]>

Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pymongo/mongo_client.py", line 1869, in _process_periodic_tasks
    self._topology.update_pool(self.__all_credentials)
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pymongo/topology.py", line 456, in update_pool
    server.pool.remove_stale_sockets(generation, all_credentials)
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pymongo/pool.py", line 1252, in remove_stale_sockets
    sock_info = self.connect(all_credentials)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pymongo/pool.py", line 1294, in connect
    sock_info.hello(all_credentials)
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pymongo/pool.py", line 552, in hello
    return self._hello(None, None,

In [ ]:
import scipy.stats as stats

workbook_name = 'Probability Of Positive Day'

# Get Data using the daily timeframe (same as the percent_mean_std page)
for opt_ndays_range in [pf.calc_percent_daily, pf.calc_percent_weekly, pf.calc_percent_2weekly, pf.calc_percent_monthly, pf.calc_percent_2monthly]:    
    ndays_range = pf.get_ndays_range(opt_ndays_range)
    worksheet_id = dct_worksheet_ids[workbook_name][opt_ndays_range]
    print(f"\nProcessing timeframe: {ndays_range} (worksheet ID: {worksheet_id})")
    
    symbols = md.get_symbols(md.all)
    # Fetch the core dataframe
    df_raw = pf.df_secind_sym_perf(ndays_range, symbols)
    df = df_raw.dropna(subset=['over_pc', 'pc_mean', 'pc_std']).copy()

    # Ensure we don't have zeros in our standard deviation (which would break division)
    df_prob = df[df['pc_std'] > 0].copy()
    
    # Calculate the Survival Function (1 - CDF) at 0.0 % using the mean and standard deviation.
    # This essentially gives us P(X > 0)
    df_prob['prob_green_day'] = stats.norm.sf(0, loc=df_prob['pc_mean'], scale=df_prob['pc_std'])
    
    # Convert to a readable percentage format
    df_prob['prob_green_day_%'] = (df_prob['prob_green_day'] * 100).round(2)
    
    # Sort by the highest probability of closing green
    df_prob.sort_values(by='prob_green_day', ascending=False, inplace=True)
    
    # Select columns to display
    display_df = df_prob[['sector', 'industry', 'symbol', 'pc_mean', 'pc_std', 'prob_green_day_%']].head(50)
    display(display_df.head(5))
    worksheet_update_with_df(workbook_name, worksheet_id, display_df)
